# Lab 4: MLPs, encoders and cross-modal latent translation

Run the complete path: `simulation geometry → encoder → latent mapper → sensor decoder → three-axis output`.

The default is five measured episodes. Switch `DOMAIN` to `synthetic` for 300 generated sequences with fixture and speed variation. Synthetic signals are not new hardware measurements; their scores describe the teaching generator.

Watch `lab4/figures/replay_comparison.gif`: left is the original replay, right is a synthetic kinematic replay with a fixed virtual fixture shift. The right-hand signal comes from an analytic teaching model.

Run all cells once before changing parameters. CPU is sufficient.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'lab4').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'lab4').is_dir(), 'Start the notebook from the repository root or lab4 directory'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab4.data import load_windows
from lab4.models import mlp, Autoencoder, LatentTranslator
from lab4.train import fit, metrics

DATA_DIR = ROOT / 'lab4/datasets'
DOMAIN = 'measured'             # Switch to synthetic for the generated dataset
WINDOW = 7                     # Current and six past packets; synthetic duration depends on speed
EPOCHS = 600
SIM_LATENT = 8
SENSOR_LATENT = 4
SEED = 7

torch.set_num_threads(2)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
data = load_windows(DATA_DIR / f'{DOMAIN}.npz', window=WINDOW)
print('Target source:', data['source_kind'])
for split in ['train', 'validation', 'test']:
    m = data['split'] == split
    print(split, 'windows=', m.sum(), 'parents=', np.unique(data['parent'][m]))


## 1. What is a paired sample?

Each instant provides 11 simulation features: sensor-local closest point (3), local normal (3), signed gap, backward gap velocity, indentation, geometric contact indicator and packet interval. A closest point is not necessarily an actual contact point.

The target is ID3's three-axis sensor output. Seven causal time steps produce 77 input dimensions and a 21-dimensional target history. Measured outputs use nominal SDK unit conversion.

Every descendant of an original episode stays in the same split. Standardization is fitted on training frames only.

In [ ]:
x = torch.from_numpy(data['x'])
y = torch.from_numpy(data['y'])
train = data['split'] == 'train'
validation = data['split'] == 'validation'
test = data['split'] == 'test'
xt, xv = x[train], x[validation]
yt, yv = y[train], y[validation]
print('All input windows:', tuple(x.shape), 'All target windows:', tuple(y.shape))
print('Features:', data['feature_names'].tolist())

seq = np.unique(data['sequence'][train])[0]
selected = data['sequence'] == seq
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(data['time'][selected], data['target'][selected])
ax.legend(['x', 'y', 'z']); ax.set(xlabel='Time (s)', ylabel='Nominal N', title=seq)
ax.grid(alpha=.2)
plt.show()


## 2. Direct MLP: predict without an explicit latent bridge

An MLP uses linear layers and nonlinear activations to map a 77-dimensional window to the current three-axis output. Train with standardized mean squared error.

Create all three networks here to reproduce the seeded reference experiment. `fit` selects an epoch on validation data, not test data.

In [ ]:
direct = mlp(x.shape[1], 3)
sim_ae = Autoencoder(x.shape[1], SIM_LATENT)
sensor_ae = Autoencoder(y.shape[1], SENSOR_LATENT)
print(direct)
direct_history = fit(direct, xt, yt[:, -3:], xv, yv[:, -3:], epochs=EPOCHS)
print('MLP selected epoch:', direct_history['best_epoch'])


## 3. Two autoencoders: reconstruct each modality

Simulation: `77 → encoder → 8 → decoder → 77`.

Sensor: `21 → encoder → 4 → decoder → 21`.

The bottleneck vector is a latent representation. Its coordinates do not automatically acquire physical meanings; independently trained latent spaces do not automatically align.

In [ ]:
sim_history = fit(sim_ae, xt, xt, xv, xv, epochs=EPOCHS)
sensor_history = fit(sensor_ae, yt, yt, yv, yv, epochs=EPOCHS)
with torch.no_grad():
    z_sim_example = sim_ae.encoder(xt[:5])
    z_sensor_example = sensor_ae.encoder(yt[:5])
print('Simulation latent:', tuple(z_sim_example.shape))
print('Sensor latent:', tuple(z_sensor_example.shape))
print('Sensor reconstruction validation loss:', sensor_history['best_validation_loss'])


## 4. Translate latents: freeze both ends, learn the middle

Training: compute `z_sim = E_sim(x_sim)` and `z_sensor = E_sensor(x_sensor)`. Learn `T(z_sim) ≈ z_sensor` while checking decoded sensor reconstruction.

Inference: input **only the simulation window**, and execute `D_sensor(T(E_sim(x_sim)))`.

Frozen encoders/decoder make the mapper's role explicit. The loss combines decoded reconstruction and a normalized latent matching term.

In [ ]:
translator = LatentTranslator(sim_ae, sensor_ae, SIM_LATENT, SENSOR_LATENT)
with torch.no_grad():
    translator.sensor_latent_scale.copy_(sensor_ae.encoder(yt).std(dim=0, unbiased=False).clamp_min(.1))
translation_history = fit(translator, xt, yt, xv, yv, epochs=EPOCHS,
                          objective=translator.objective)
print('Trainable parameters:', sum(p.numel() for p in translator.parameters() if p.requires_grad))
print('Latent mapping selected epoch:', translation_history['best_epoch'])


## 5. Test on the held-out original episode

Compare a training-mean baseline, direct MLP and latent translation. Sensor self-reconstruction is also shown, but it reads the target at inference: it is a compression reference, not a cross-modal predictor.

Undo standardization before reporting nominal N errors. The test set has only one original motion; this is a classroom comparison, not cross-object validation.

In [ ]:
scaler = data['y_scaler']
with torch.no_grad():
    pred_direct = scaler.inverse(direct(x).numpy())
    pred_translated = scaler.inverse(translator(x).numpy()[:, -3:])
    pred_self = scaler.inverse(sensor_ae(y).numpy()[:, -3:])
pred_mean = np.broadcast_to(data['target'][train].mean(axis=0), data['target'].shape)
for name, pred in [('Training mean', pred_mean), ('Direct MLP', pred_direct),
                   ('Latent translation', pred_translated), ('Sensor self-reconstruction (uses target)', pred_self)]:
    result = metrics(pred[test], data['target'][test])
    print(name, 'RMSE=', round(result['rmse'], 4), 'per-axis MAE=', np.round(result['per_axis_mae'], 4))

seq = np.unique(data['sequence'][test])[0]
selected = data['sequence'] == seq
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for axis, ax in enumerate(axes):
    ax.plot(data['time'][selected], data['target'][selected, axis], 'k', label='Target')
    ax.plot(data['time'][selected], pred_direct[selected, axis], label='Direct MLP')
    ax.plot(data['time'][selected], pred_translated[selected, axis], label='Latent translation')
    ax.set_ylabel('xyz'[axis] + ' (nominal N)'); ax.grid(alpha=.2)
axes[0].legend(); axes[0].set_title(DOMAIN + ' | held-out episode')
axes[-1].set_xlabel('Synthetic sequence time (s)' if DOMAIN == 'synthetic' else 'Trigger-relative time (s)')
fig.tight_layout()
plt.show()


## 6. Inspect the latent space

In sensor latent coordinates, circles denote encoded targets and crosses denote predictions mapped from simulation. Both use the same PCA projection, fitted on training target latents only.

Do not independently project two spaces and claim alignment merely because the resulting pictures look similar.

In [ ]:
with torch.no_grad():
    zr = sensor_ae.encoder(y).numpy()
    zm = translator.latent(x).numpy()
center = zr[train].mean(axis=0)
_, _, vt = np.linalg.svd(zr[train] - center, full_matrices=False)
basis = vt[:2].T
r = (zr[selected] - center) @ basis
m = (zm[selected] - center) @ basis
if r.shape[1] == 1:
    r = np.column_stack([r, np.zeros(len(r))])
    m = np.column_stack([m, np.zeros(len(m))])
color = np.linalg.norm(data['target'][selected], axis=1)
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(r[:, 0], r[:, 1], c=color, label='Sensor latent', cmap='viridis')
ax.scatter(m[:, 0], m[:, 1], c=color, marker='x', label='Mapped sim latent', cmap='viridis')
for a, b in zip(r[::3], m[::3]):
    ax.plot([a[0], b[0]], [a[1], b[1]], color='gray', alpha=.3)
ax.set(xlabel='Training PCA axis 1', ylabel='Training PCA axis 2', title=DOMAIN)
ax.legend(); fig.colorbar(sc, ax=ax, label='Target norm (nominal N)')
plt.show()


## 7. Exercises and submission

1. Change `WINDOW` from 7 to 1 and rerun. Why might sensor lag make a single-frame predictor worse?
2. Try `SENSOR_LATENT` values 2, 4 and 8. Compare self-reconstruction and cross-modal errors. Which stage is limiting performance?
3. Switch `DOMAIN` to `synthetic`. Which scores describe the generator, and which come from measured targets?
4. Compare direct MLP and latent translation. Why do additional stages not guarantee better accuracy?
5. Inspect the command-line ridge baseline. What does a complex model add when a linear model already works well?
6. Explain why 300 synthetic sequences still have five original motion sources and must not be split randomly by frame.

Submit a three-axis prediction plot, a latent plot, a parameter/error table and answers to at least two questions. Tune on validation data, then use test data for a final report. Repeated test-driven tuning turns that test set into development data.

Latent translation learns a relationship between observations; it does not change physical reality or turn simulated signals into hardware measurements.